In [1]:
"""Basic Customer Churn Prediction using an ANN."""

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Make results repeatable
np.random.seed(42)
tf.keras.utils.set_random_seed(42)

# 1. Create a practice dataset
rows = 1000
rng = np.random.default_rng(42)

data = pd.DataFrame({
    "tenure": rng.integers(1, 73, rows),
    "monthly_bill": rng.integers(20, 121, rows),
    "contract": rng.choice(["Monthly", "Yearly"], rows),
})

# Monthly customers and newer customers are more likely to leave
chance = (
    0.15
    + 0.35 * (data["contract"] == "Monthly")
    + 0.15 * (data["tenure"] < 12)
)
data["churn"] = rng.binomial(1, chance.clip(0, 0.9))

# 2. Prepare the data
# Convert words into numeric columns
X = pd.get_dummies(data.drop("churn", axis=1), dtype=int)
y = data["churn"]

# Split: 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features to similar ranges
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 3. Build the ANN
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

# 4. Train the model
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
)

# 5. Predict churn for the test customers
probability = model.predict(X_test, verbose=0).flatten()
prediction = (probability >= 0.5).astype(int)

print("\nTest accuracy:", round(accuracy_score(y_test, prediction), 3))
print("First 10 predictions (0 = stay, 1 = leave):")
print(prediction[:10])

# 6. Draw text-based curves using NumPy and Python
# Horizontal position is the epoch; vertical position is the metric value.
def show_curve(values, name):
    values = np.array(values, dtype=float)
    height = 10
    spread = np.ptp(values)

    if spread == 0:
        y_points = np.zeros(len(values), dtype=int)
    else:
        y_points = np.rint(
            (values - values.min()) / spread * (height - 1)
        ).astype(int)

    graph = [[" " for _ in values] for _ in range(height)]

    # Connect each epoch's point to the next one
    for x in range(len(values) - 1):
        y1 = y_points[x]
        y2 = y_points[x + 1]
        steps = max(abs(y2 - y1), 1)

        for step in range(steps + 1):
            column = round(x + step / steps)
            row = height - 1 - round(y1 + (y2 - y1) * step / steps)
            graph[row][column] = "*"

    print(f"\n{name} (vertical: value, horizontal: epoch)")
    for row in graph:
        print("|" + "".join(row))

    print("+" + "-" * len(values) + "> epoch")
    print(f"Values range from {values.min():.2f} to {values.max():.2f}")

# Show training and validation accuracy and loss
show_curve(history.history["accuracy"], "Training accuracy")
show_curve(history.history["val_accuracy"], "Validation accuracy")
show_curve(history.history["loss"], "Training loss")
show_curve(history.history["val_loss"], "Validation loss")

Epoch 1/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.5063 - loss: 0.6779 - val_accuracy: 0.5250 - val_loss: 0.6986
Epoch 2/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6219 - loss: 0.6544 - val_accuracy: 0.6500 - val_loss: 0.6823
Epoch 3/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6734 - loss: 0.6379 - val_accuracy: 0.6500 - val_loss: 0.6683
Epoch 4/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6828 - loss: 0.6258 - val_accuracy: 0.6313 - val_loss: 0.6565
Epoch 5/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6781 - loss: 0.6180 - val_accuracy: 0.6313 - val_loss: 0.6476
Epoch 6/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6922 - loss: 0.6124 - val_accuracy: 0.6313 - val_loss: 0.6399
Epoch 7/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6891 - loss: 0.6078 - val_accuracy: 0.6438 - val_loss: 0.6332
Epoch 8/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6922 - loss: 0.6039 - val_accuracy: 0.6438 - val_loss